# 10 — Export the saved results chapter package

Run this after all **24 confirmation streams** finish. It authenticates saved results, recovers valid completion bookkeeping when needed, and creates a compact ZIP for thesis writing. It does not train, load a checkpoint, generate images or compute new predictions.

Use a **TensorFlow 2.20 / Keras 3** kernel.

In [ ]:
import os
import sys
from pathlib import Path
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "semantic_consolidation/config.py").is_file())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from notebooks.thesis.workflow import check_runtime
print(check_runtime())
import json
import pandas as pd
from IPython.display import FileLink
from notebooks.thesis.results_package import export_results_package

The final package requires three paired seeds `[1103, 2207, 3301]`, full five/ten-task schedules and every declared condition. Keep the development seed 17 and older two-seed campaigns separate. `PROGRESS = True` creates an explicitly marked progress package without native final inference; use it only to inspect already completed streams. If a progress snapshot already exists with fewer completed runs, choose a new `OUTPUT` directory to preserve it.

In [ ]:
CAMPAIGN = ROOT / "results/thesis_route_one/minimum_v4_tf220"
PROGRESS = False
DETAILS = False  # True adds optional diagnostic tables and figures; choose a new OUTPUT.
OUTPUT = None  # Default: writing_package (final) or progress_package.
package = export_results_package(CAMPAIGN / "frozen_design.json",
                                 progress=PROGRESS, output_dir=OUTPUT, details=DETAILS)
print("Package:", package["directory"])
print("ZIP:", package["zip"])
display(FileLink(str(package["zip"])))

Inspect the main results and primary paired effects. All comparable outcomes are computed within each stream first, then summarized with mean, sample SD (`ddof=1`) and actual n. Blank observations remain unavailable. The native primary 95% paired interval is retained separately from SD.

The compact table includes local backward transfer (final minus acquisition accuracy), which differs from TMCL's reference-model transfer metric.

In [ ]:
summary = json.loads((package["directory"] / "RESULT_SUMMARY.json").read_text("utf-8"))
print(summary["status"], "—", summary["completed_streams"], "completed streams")
for name in ("thesis_summary",):
    print(name.replace("_", " "))
    display(pd.DataFrame(summary["tables"][name]))
if summary["native_primary_statistics"]:
    display(pd.read_csv(package["directory"] / "tables/T90_primary_native_interval.csv"))
else:
    print("Progress view: no final paired confidence interval yet.")

Read the compact treatment summary and the primary learned-minus-extra-joint paired interval first. The default ZIP retains full numeric source rows, configuration, run identities and hashes. Set `DETAILS=True` with a new output directory only when the additional validation diagnostics, trajectories and saved replay figures are useful for writing.

Every complete stream is an independent replicate; three pairs give limited precision. Test outcomes describe efficacy, validation measurements describe mechanisms, and replay images are qualitative. Preserve negative and unavailable outcomes. Collection never trains or changes original results. Write from the saved observations and retain these limits.